# <b><font color='cornflowerblue'>Сюжет игры</font></b>
Ночью из дата-центра исчез файл.

Логи повреждены. Известно только, что:
- саботаж совершил сотрудник компании;
- он оставил следы в данных;

Восстановить доступ можно, если определить:
- город
- отдел
- сотрудника
- устройство
- дату атаки
- код операции

Из этих шести частей собирается пароль.
Пароль является кодом доступа к исчезнувшему файлу.

<b> Задача: </b> Восстановить пароль и получить доступ к файлу.

## <b><font color='cornflowerblue'>Данные</font></b>
- employees.csv
- devices.csv
- incidents.csv
- access_logs.csv

In [63]:
import pandas as pd
import numpy as np


In [64]:
# - employees.csv
# - devices.csv
# - incidents.csv
# - access_logs.csv

employees = pd.read_csv('..//data//employees.csv')
display(employees)
devices = pd.read_csv('..//data//devices.csv')
display(devices)
incidents = pd.read_csv('..//data//incidents.csv')
display(incidents)
access_logs = pd.read_csv('..//data//access_logs.csv')
display(access_logs)


,Unnamed: 0,employee_id,name,city,department,experience,salary
0,0,1151,James Miller,Berlin,Security,12,125000
1,1,1002,Benjamin Davis,Rome,Security,8,94392
2,2,1003,Henry Thomas,Madrid,IT,1,92966
3,3,1004,Emma White,Rome,Finance,13,99325
4,4,1005,Sophia Harris,Paris,IT,10,84230
...,...,...,...,...,...,...,...
145,145,1146,James Young,Madrid,IT,6,76279
146,146,1147,Oliver Miller,Rome,Operations,13,86227
147,147,1148,Mia Walker,Madrid,Finance,10,128901
148,148,1149,Ethan Thomas,Madrid,IT,9,93459


,Unnamed: 0,device_id,device_type,security_level,location
0,0,1,SERVER-01,medium,London
1,1,2,SERVER-02,low,Madrid
2,2,3,SERVER-03,critical,Rome
3,3,4,SERVER-04,critical,Rome
4,4,5,SERVER-05,medium,Amsterdam
5,5,6,SERVER-06,low,Amsterdam
6,6,7,SERVER-07,critical,Berlin
7,7,8,SERVER-08,high,Madrid
8,8,9,SERVER-09,medium,Rome
9,9,10,SERVER-10,high,Berlin


,Unnamed: 0,employee_id,incident_type,severity,date
0,0,1124,privilege_escalation,high,2026-05-07
1,1,1083,policy_violation,low,2026-05-06
2,2,1134,policy_violation,medium,2026-05-21
3,3,1081,data_copy,low,2026-05-19
4,4,1002,privilege_escalation,high,2026-05-14
...,...,...,...,...,...
520,520,1051,policy_violation,high,2026-05-13
521,521,1051,data_copy,high,2026-05-11
522,522,1051,privilege_escalation,high,2026-05-12
523,523,1051,policy_violation,high,2026-05-11


,Unnamed: 0,employee_id,device_id,date,action,duration
0,0,1044,20,2026-05-08,DELETE,14
1,1,1147,26,2026-05-01,LOGIN,278
2,2,1049,27,2026-05-31,DELETE,59
3,3,1091,14,2026-05-01,DELETE,415
4,4,1081,36,2026-05-23,EXPORT,316
...,...,...,...,...,...,...
2555,2555,1051,7,2026-05-14,LOGIN,60
2556,2556,1051,7,2026-05-14,LOGIN,177
2557,2557,1051,7,2026-05-14,LOGIN,178
2558,2558,1051,7,2026-05-14,LOGIN,58


## <b><font color='cornflowerblue'>Глава 1. Подозрительные города</font></b>
<u>Система сообщает:</u> Атака произошла из города с максимальным числом инцидентов высокой критичности.

<i>1 часть пароля:</i> 3 первых буквы получившегося города

In [65]:
employees[["experience", "city"]].max()
# Rom

experience      14
city          Rome
dtype: object

In [66]:
employees

,Unnamed: 0,employee_id,name,city,department,experience,salary
0,0,1151,James Miller,Berlin,Security,12,125000
1,1,1002,Benjamin Davis,Rome,Security,8,94392
2,2,1003,Henry Thomas,Madrid,IT,1,92966
3,3,1004,Emma White,Rome,Finance,13,99325
4,4,1005,Sophia Harris,Paris,IT,10,84230
...,...,...,...,...,...,...,...
145,145,1146,James Young,Madrid,IT,6,76279
146,146,1147,Oliver Miller,Rome,Operations,13,86227
147,147,1148,Mia Walker,Madrid,Finance,10,128901
148,148,1149,Ethan Thomas,Madrid,IT,9,93459


In [67]:
incidents_2 = incidents.merge(employees, on = 'employee_id', how = 'left')
display(incidents_2)

,Unnamed: 0_x,employee_id,incident_type,severity,date,Unnamed: 0_y,name,city,department,experience,salary
0,0,1124,privilege_escalation,high,2026-05-07,123,Amelia Miller,Rome,Security,11,119063
1,1,1083,policy_violation,low,2026-05-06,82,Oliver Brown,Berlin,Security,1,108700
2,2,1134,policy_violation,medium,2026-05-21,133,Emma Hall,Paris,IT,3,82609
3,3,1081,data_copy,low,2026-05-19,80,Ethan King,Amsterdam,Security,9,67530
4,4,1002,privilege_escalation,high,2026-05-14,1,Benjamin Davis,Rome,Security,8,94392
...,...,...,...,...,...,...,...,...,...,...,...
520,520,1051,policy_violation,high,2026-05-13,50,James Walker,London,HR,11,101164
521,521,1051,data_copy,high,2026-05-11,50,James Walker,London,HR,11,101164
522,522,1051,privilege_escalation,high,2026-05-12,50,James Walker,London,HR,11,101164
523,523,1051,policy_violation,high,2026-05-11,50,James Walker,London,HR,11,101164


In [68]:
incidents_2_g = incidents_2.groupby(['severity', 'city']).agg({'employee_id': 'count'}).sort_values('employee_id', ascending = False)
display(incidents_2_g)
# Lon

employee_id
severity city                  
high     London              40
         Berlin              38
medium   Amsterdam           37
high     Madrid              36
low      Madrid              35
         Berlin              33
         Rome                32
high     Rome                32
         Amsterdam           31
medium   Berlin              31
low      London              30
         Amsterdam           29
         Paris               29
high     Paris               20
medium   Madrid              20
         Paris               20
         London              16
         Rome                16

## <b><font color='cornflowerblue'>Глава 2. Внутренний агент</font></b>
<u>Система сообщает:</u> Подозреваемый работает в отделе, где средняя длительность доступа к системе была максимальной.

<i>2 часть пароля:</i> 2 первых буквы названия отдела

In [69]:
access_logs

,Unnamed: 0,employee_id,device_id,date,action,duration
0,0,1044,20,2026-05-08,DELETE,14
1,1,1147,26,2026-05-01,LOGIN,278
2,2,1049,27,2026-05-31,DELETE,59
3,3,1091,14,2026-05-01,DELETE,415
4,4,1081,36,2026-05-23,EXPORT,316
...,...,...,...,...,...,...
2555,2555,1051,7,2026-05-14,LOGIN,60
2556,2556,1051,7,2026-05-14,LOGIN,177
2557,2557,1051,7,2026-05-14,LOGIN,178
2558,2558,1051,7,2026-05-14,LOGIN,58


In [70]:
employees

,Unnamed: 0,employee_id,name,city,department,experience,salary
0,0,1151,James Miller,Berlin,Security,12,125000
1,1,1002,Benjamin Davis,Rome,Security,8,94392
2,2,1003,Henry Thomas,Madrid,IT,1,92966
3,3,1004,Emma White,Rome,Finance,13,99325
4,4,1005,Sophia Harris,Paris,IT,10,84230
...,...,...,...,...,...,...,...
145,145,1146,James Young,Madrid,IT,6,76279
146,146,1147,Oliver Miller,Rome,Operations,13,86227
147,147,1148,Mia Walker,Madrid,Finance,10,128901
148,148,1149,Ethan Thomas,Madrid,IT,9,93459


In [71]:
access_logs_2 = access_logs.merge(employees, on = 'employee_id', how = 'left')
display(access_logs_2.sort_values('duration', ascending = False).head(10))

,Unnamed: 0_x,employee_id,device_id,date,action,duration,Unnamed: 0_y,name,city,department,experience,salary
1298,1298,1017,7,2026-05-25,UPLOAD,499,16,Grace Miller,Rome,HR,14,79565
1303,1303,1081,25,2026-05-14,EXPORT,499,80,Ethan King,Amsterdam,Security,9,67530
1438,1438,1070,23,2026-05-17,EXPORT,499,69,Benjamin Anderson,Berlin,HR,2,96818
2355,2355,1119,36,2026-05-29,DOWNLOAD,499,118,James Thomas,Paris,IT,3,108265
1952,1952,1133,2,2026-05-27,EXPORT,499,132,James Wilson,Amsterdam,HR,1,64216
859,859,1064,1,2026-05-06,LOGIN,498,63,Oliver Wilson,London,Security,1,97065
1941,1941,1067,18,2026-05-23,DELETE,498,66,Sophia Hall,Amsterdam,Finance,6,61974
1608,1608,1144,14,2026-05-12,DELETE,498,143,Henry Thomas,Paris,Operations,4,63176
746,746,1055,19,2026-05-06,UPLOAD,497,54,Sophia Walker,Madrid,HR,12,81289
2249,2249,1002,11,2026-05-24,DOWNLOAD,497,1,Benjamin Davis,Rome,Security,8,94392


In [72]:
logs_with_emp = access_logs.merge(
employees,
on='employee_id',
how='left'
)

department_mean = logs_with_emp.groupby('department')['duration'].mean()

print(department_mean)

part_2 = department[:2].upper()

print(department)
print(part_2)

department
Finance       251.983287
HR            237.210111
IT            256.261698
Operations    250.448357
Security      249.208850
Name: duration, dtype: float64


NameError: name 'department' is not defined

## <b><font color='cornflowerblue'>Глава 3. Кто именно?</font></b>
Нужно найти сотрудника, из города в первой главе, отдела во второй главе и с максимальным количеством инцидентов.

<i>3 часть пароля:</i> 3 первых буквы имени сотрудника

In [ ]:
employees

,Unnamed: 0,employee_id,name,city,department,experience,salary
0,0,1151,James Miller,Berlin,Security,12,125000
1,1,1002,Benjamin Davis,Rome,Security,8,94392
2,2,1003,Henry Thomas,Madrid,IT,1,92966
3,3,1004,Emma White,Rome,Finance,13,99325
4,4,1005,Sophia Harris,Paris,IT,10,84230
...,...,...,...,...,...,...,...
145,145,1146,James Young,Madrid,IT,6,76279
146,146,1147,Oliver Miller,Rome,Operations,13,86227
147,147,1148,Mia Walker,Madrid,Finance,10,128901
148,148,1149,Ethan Thomas,Madrid,IT,9,93459


In [ ]:
incidents_2 = incidents.merge(employees, on = 'employee_id', how = 'left')
incidents_2_c = incidents_2[incidents_2['city'] == 'London']
incidents_2_d = incidents_2_c[incidents_2_c['department'] == 'IT']
display(incidents_2_d)


,Unnamed: 0_x,employee_id,incident_type,severity,date,Unnamed: 0_y,name,city,department,experience,salary
68,68,1063,privilege_escalation,low,2026-05-04,62,Mia Walker,London,IT,10,54236
104,104,1143,unauthorized_access,low,2026-05-14,142,Noah Young,London,IT,10,95728
122,122,1143,privilege_escalation,high,2026-05-22,142,Noah Young,London,IT,10,95728
212,212,1063,policy_violation,high,2026-05-23,62,Mia Walker,London,IT,10,54236
260,260,1143,unauthorized_access,low,2026-05-05,142,Noah Young,London,IT,10,95728
285,285,1143,policy_violation,high,2026-05-08,142,Noah Young,London,IT,10,95728
291,291,1139,data_copy,medium,2026-05-03,138,Charlotte Harris,London,IT,2,112344
321,321,1139,data_copy,low,2026-05-20,138,Charlotte Harris,London,IT,2,112344
335,335,1063,unauthorized_access,low,2026-05-08,62,Mia Walker,London,IT,10,54236
350,350,1143,privilege_escalation,low,2026-05-21,142,Noah Young,London,IT,10,95728


In [ ]:
inc_count = incidents_2_d.groupby('employee_id').size().sort_values(ascending=False)
display(inc_count)

employee_id
1143    6
1063    3
1139    2
dtype: int64

In [ ]:
name = incidents_2_d[incidents_2_d['employee_id'] == 1143]['name'].values[0]
print(name)
Noa

Noah Young


## <b><font color='cornflowerblue'>Глава 4. Орудие преступления</font></b>
Атака была проведена с устройства, которое чаще всего использовалось подозреваемым.

<i>4 часть пароля:</i> номер устройства (последние 2 символа имени устройства)

In [ ]:
devices

,Unnamed: 0,device_id,device_type,security_level,location
0,0,1,SERVER-01,medium,London
1,1,2,SERVER-02,low,Madrid
2,2,3,SERVER-03,critical,Rome
3,3,4,SERVER-04,critical,Rome
4,4,5,SERVER-05,medium,Amsterdam
5,5,6,SERVER-06,low,Amsterdam
6,6,7,SERVER-07,critical,Berlin
7,7,8,SERVER-08,high,Madrid
8,8,9,SERVER-09,medium,Rome
9,9,10,SERVER-10,high,Berlin


In [77]:
noah_inc_dates = incidents_2_d[incidents_2_d['employee_id'] == top_emp_id][['employee_id', 'date']]
print(noah_inc_dates)
# Мержим его логи с датами инцидентов
emp_logs_crime = emp_logs.merge(noah_inc_dates, on=['employee_id', 'date'])
display(emp_logs_crime)
# Теперь считаем устройства
device_counts = emp_logs_crime.groupby('device_id').size().sort_values(ascending=False)
print(device_counts)

     employee_id        date
104         1143  2026-05-14
122         1143  2026-05-22
260         1143  2026-05-05
285         1143  2026-05-08
350         1143  2026-05-21
498         1143  2026-05-24


,Unnamed: 0,employee_id,device_id,date,action,duration
0,126,1143,13,2026-05-05,LOGIN,321
1,771,1143,6,2026-05-14,LOGIN,123
2,1720,1143,12,2026-05-22,EXPORT,16
3,1902,1143,14,2026-05-14,DELETE,201
4,2230,1143,11,2026-05-08,DELETE,185


device_id
6     1
11    1
12    1
13    1
14    1
dtype: int64


In [78]:
noah_inc_dates = incidents_2_d[incidents_2_d['employee_id'] == top_emp_id][['employee_id', 'date']]
print(noah_inc_dates)

     employee_id        date
104         1143  2026-05-14
122         1143  2026-05-22
260         1143  2026-05-05
285         1143  2026-05-08
350         1143  2026-05-21
498         1143  2026-05-24


In [81]:
emp_logs = access_logs[access_logs['employee_id'] == 1143]
display(emp_logs)
emp_logs_d = emp_logs.merge(devices, on=['device_id'])
emp_logs_d

,Unnamed: 0,employee_id,device_id,date,action,duration
77,77,1143,17,2026-05-16,LOGIN,164
126,126,1143,13,2026-05-05,LOGIN,321
239,239,1143,10,2026-05-29,DOWNLOAD,128
287,287,1143,20,2026-05-01,LOGIN,302
429,429,1143,23,2026-05-19,EXPORT,445
443,443,1143,19,2026-05-09,UPLOAD,376
474,474,1143,4,2026-05-27,LOGIN,440
603,603,1143,3,2026-05-31,EXPORT,99
635,635,1143,18,2026-05-18,UPLOAD,290
771,771,1143,6,2026-05-14,LOGIN,123


,Unnamed: 0_x,employee_id,device_id,date,action,duration,Unnamed: 0_y,device_type,security_level,location
0,77,1143,17,2026-05-16,LOGIN,164,16,SERVER-17,critical,Berlin
1,126,1143,13,2026-05-05,LOGIN,321,12,SERVER-13,medium,Berlin
2,239,1143,10,2026-05-29,DOWNLOAD,128,9,SERVER-10,high,Berlin
3,287,1143,20,2026-05-01,LOGIN,302,19,SERVER-20,critical,Rome
4,429,1143,23,2026-05-19,EXPORT,445,22,LAPTOP-23,medium,Madrid
5,443,1143,19,2026-05-09,UPLOAD,376,18,SERVER-19,critical,Paris
6,474,1143,4,2026-05-27,LOGIN,440,3,SERVER-04,critical,Rome
7,603,1143,3,2026-05-31,EXPORT,99,2,SERVER-03,critical,Rome
8,635,1143,18,2026-05-18,UPLOAD,290,17,SERVER-18,high,Madrid
9,771,1143,6,2026-05-14,LOGIN,123,5,SERVER-06,low,Amsterdam


In [74]:
device_counts = emp_logs.groupby('device_id').size().sort_values(ascending=False)
print(device_counts)

device_id
3     2
8     2
6     2
2     1
4     1
9     1
10    1
11    1
12    1
13    1
14    1
15    1
17    1
18    1
19    1
20    1
23    1
24    1
26    1
31    1
34    1
dtype: int64


In [ ]:
top_device_id = device_counts.idxmax()
print(top_device_id)

3


In [79]:
device_name_1 = devices[devices['device_id'] == 3]['device_type'].values[0]
device_name_2 = devices[devices['device_id'] == 8]['device_type'].values[0]
device_name_3 = devices[devices['device_id'] == 6]['device_type'].values[0]
print(device_name_1, device_name_2, device_name_3)

SERVER-03 SERVER-08 SERVER-06


## <b><font color='cornflowerblue'>Глава 5. День атаки</font></b>
Файл был украден в день максимальной активности подозреваемого. 
Подсказка: ориентируйтесь на таблицу логов

<i>5 часть пароля:</i> день (например, если это 15 мая, то берем 15)

In [75]:
# считаем, сколько действий подозреваемый совершил в каждый день
day_counts = suspect_logs['date'].value_counts()

# смотрим статистику по дням
print(day_counts)

# берём дату с максимальным количеством действий
attack_date = day_counts.idxmax()

# из даты формата YYYY-MM-DD берём последние 2 символа (день месяца)
part_5 = attack_date[-2:]

print(attack_date)
print(part_5)

NameError: name 'suspect_logs' is not defined

## <b><font color='cornflowerblue'>Глава 6. Код операции</font></b>
Нужно:
- оставить только действия подозреваемого в день атаки;
- посчитать частоты действий;
- отсортировать;
- взять второе по популярности действие;

<i>6 часть пароля:</i> первые три символа названия операции

## <b><font color='cornflowerblue'>Финал. Пароль</font></b>
Соберите все части пароля:
- заглавные буквы
- части пароля разделяются символом: "-"
- получившийся пароль - код доступа к архиву "Пропавший файл.zip"
- если Вы выполнили все задания верно, то архив откроется